# Stage 1 — Foundation Models across Time Granularities (Monash Energy)

**Goal.** See how three pretrained time-series foundation models — **Chronos** (Amazon),
**TimesFM** (Google) and **Moirai** (Salesforce) — behave on Monash **energy** series at
different temporal resolutions (**minutes → hours → day**), and **compare the three
models** against each other. **No baselines** — this is purely about the foundation models.

**Method.** For each energy type (electricity demand, solar, wind) we take one
high-resolution Monash series and **resample it** to each granularity. Feeding the *same*
underlying signal at different resolutions means any accuracy difference is due to
granularity — not to a different dataset. Each model forecasts **zero-shot**; we score with
MASE, sMAPE, RMSE, MAE and inference time, then compare the models per granularity and per
energy type.

> **Runtime.** A GPU makes this much faster but is **not required** — the code auto-detects
> CUDA and falls back to CPU (slower, but fine for the small default batch).
>
> **Running in VS Code / local Jupyter.** This notebook runs as-is. Notes:
> the `#@param ... { display-mode: "form" }` comments are Colab-only — in VS Code they
> render as ordinary comments and the variables still take their assigned values (edit them
> directly). Before running, create an environment and select it as the kernel
> (see the `requirements.txt` / setup steps shipped alongside this notebook). The `%pip
> install` cells work inside a Jupyter kernel; if you prefer, install from `requirements.txt`
> in a terminal instead and skip the install cells.


## 1. Install & check runtime

In [2]:
#@title Install dependencies (Chronos + TimesFM + Moirai) { display-mode: "form" }
INSTALL_CHRONOS = True   #@param {type:"boolean"}
INSTALL_TIMESFM = True   #@param {type:"boolean"}
INSTALL_MOIRAI  = True   #@param {type:"boolean"}

%pip install -q "datasets>=2.19" pandas pyarrow matplotlib tqdm 2>/dev/null
if INSTALL_CHRONOS:
    %pip install -q chronos-forecasting 2>/dev/null
if INSTALL_TIMESFM:
    %pip install -q "timesfm[torch]" 2>/dev/null
if INSTALL_MOIRAI:
    %pip install -q uni2ts 2>/dev/null
print("Installs done. If TimesFM/Moirai raise dependency errors on import below, "
      "restart the runtime (Runtime > Restart session) and re-run from here.")


Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.


Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.


Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.


Note: you may need to restart the kernel to use updated packages.
Installs done. If TimesFM/Moirai raise dependency errors on import below, restart the runtime (Runtime > Restart session) and re-run from here.


The system cannot find the path specified.


In [4]:
#@title GPU check
import torch
print("PyTorch:", torch.__version__)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — enable GPU for reasonable speed.")


PyTorch: 2.5.1+cu121
GPU    : NVIDIA GeForce RTX 3060 Laptop GPU


## 2. Configuration

A small batch keeps the whole comparison quick; raise `SERIES_PER_TYPE` once it runs.
The granularity ladder is **minutes (native) → hourly → daily**.


In [1]:
#@title Config
import numpy as np
np.random.seed(7)

# One base Monash series per energy type (native = finest resolution), then resample.
BASE = {
    "load":  {"config": "australian_electricity_demand", "native_freq": "30min", "label": "AU Electricity Demand"},
    "solar": {"config": "solar_10_minutes",              "native_freq": "10min", "label": "Solar Generation"},
    "wind":  {"config": "wind_farms_minutely",           "native_freq": "1min",  "label": "Wind Generation"},
}

# Granularity ladder: minutes / hours / day. (pandas offset alias, seasonal m, horizon steps)
GRANULARITIES = {
    "native": {"label": "minutes (native)", "resample": None, "m": None, "horizon": 48},
    "1h":     {"label": "hourly",           "resample": "1h", "m": 24,  "horizon": 24},
    "1D":     {"label": "daily",            "resample": "1D", "m": 7,   "horizon": 14},
}
NATIVE_M = {"load": 48, "solar": 144, "wind": 1440}   # steps/day at native resolution (for MASE)

SERIES_PER_TYPE = 3      # small batch
N_TEST_WINDOWS  = 2      # rolling-origin windows per (series, granularity)
CONTEXT_LENGTH  = 512    # history length fed to each model
RANDOM_SEED     = 7
RESAMPLE_AGG    = "mean" # average power when coarsening (use 'sum' for energy totals)

RUN_MODELS = {"chronos": True, "timesfm": True, "moirai": True}
CHRONOS_MODEL = "amazon/chronos-bolt-small"
TIMESFM_REPO  = "google/timesfm-2.0-500m-pytorch"
MOIRAI_REPO   = "Salesforce/moirai-1.1-R-small"
RESULTS_PATH  = "stage1_granularity_results.parquet"

print("Types      :", list(BASE))
print("Granularity:", [g["label"] for g in GRANULARITIES.values()])
print("Models     :", [k for k,v in RUN_MODELS.items() if v])


Types      : ['load', 'solar', 'wind']
Granularity: ['minutes (native)', 'hourly', 'daily']
Models     : ['chronos', 'timesfm', 'moirai']


## 3. Load a small batch of Monash energy series

In [2]:
from datasets import load_dataset
import pandas as pd

def load_monash(config):
    ds = load_dataset("Monash-University/monash_tsf", config, trust_remote_code=True)
    split = "test" if "test" in ds else list(ds.keys())[0]
    out = []
    for row in ds[split]:
        arr = np.asarray(row["target"], float); arr = arr[np.isfinite(arr)]
        if arr.size: out.append({"target": arr, "start": row.get("start", None)})
    return out

def to_series(entry, freq):
    try:
        idx = pd.date_range(pd.Timestamp(entry["start"]), periods=len(entry["target"]), freq=freq)
    except Exception:
        idx = pd.date_range("2015-01-01", periods=len(entry["target"]), freq=freq)
    return pd.Series(entry["target"], index=idx)

min_native = CONTEXT_LENGTH + GRANULARITIES["native"]["horizon"] * N_TEST_WINDOWS
DATA = {}
for et, info in BASE.items():
    print(f"Loading {info['config']} ({et}) ...", end=" ", flush=True)
    raw = load_monash(info["config"])
    elig = [e for e in raw if e["target"].size >= min_native] or sorted(raw, key=lambda e:-e["target"].size)
    rng = np.random.default_rng(RANDOM_SEED)
    pick = [elig[i] for i in rng.permutation(len(elig))[:SERIES_PER_TYPE]]
    DATA[et] = [to_series(e, info["native_freq"]) for e in pick]
    print(f"{len(raw)} series -> using {len(DATA[et])}")


e:\Thesis\EnergyForcastModel\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading australian_electricity_demand (load) ... 5 series -> using 3
Loading solar_10_minutes (solar) ... 137 series -> using 3
Loading wind_farms_minutely (wind) ... 339 series -> using 3


In [3]:
# First rows of all sampled load series side by side
load_df = pd.concat({f"series_{i}": s for i, s in enumerate(DATA["load"])}, axis=1)
display(load_df.head(10))

,series_0,series_1,series_2
2002-01-01 00:00:00,3382.041260,5714.044922,315.915497
2002-01-01 00:30:00,3288.315674,5360.188965,306.245850
2002-01-01 01:00:00,3172.329102,5014.834961,305.762573
2002-01-01 01:30:00,3020.312988,4602.755371,295.602203
2002-01-01 02:00:00,2918.082764,4285.179688,290.447083
2002-01-01 02:30:00,2839.923828,4074.894531,282.577576
2002-01-01 03:00:00,2810.689209,3942.936035,277.692078
2002-01-01 03:30:00,2768.326660,3883.997559,275.753876
2002-01-01 04:00:00,2738.259033,3877.679932,278.291840
2002-01-01 04:30:00,2727.975586,3837.716553,272.943848


## 4. Resample, cut evaluation windows, define metrics

MASE's denominator is a seasonal-naïve scale on the context — used **only** as a
normaliser so errors are comparable across granularities, not as a baseline model.


In [6]:
from dataclasses import dataclass

def resample_series(s, rule, agg="mean"):
    if rule is None: return s
    r = s.resample(rule); return (r.mean() if agg=="mean" else r.sum()).dropna()

@dataclass
class Window:
    etype:str; series_id:int; gran:str; horizon:int
    context:np.ndarray; truth:np.ndarray; m:int

def build_windows(data):
    W=[]
    for et, series_list in data.items():
        for sid, sn in enumerate(series_list):
            for gk, g in GRANULARITIES.items():
                vals = resample_series(sn, g["resample"], RESAMPLE_AGG).values.astype(float)
                h = g["horizon"]; m = g["m"] if g["m"] is not None else NATIVE_M[et]
                for w in range(N_TEST_WINDOWS):
                    end = len(vals)-w*h; st = end-h
                    ctx = min(CONTEXT_LENGTH, st)              # adaptive: fit short coarse series
                    if ctx < max(2*h, 16): continue
                    sc = st-ctx
                    W.append(Window(et, sid, gk, h, vals[sc:st].copy(),
                                    vals[st:end].copy(), min(m, ctx-1)))
    return W

WINDOWS = build_windows(DATA)
from collections import Counter
print(len(WINDOWS), "windows |", dict(Counter(w.gran for w in WINDOWS)))


54 windows | {'native': 18, '1h': 18, '1D': 18}


In [7]:
def _scale(ctx, m):
    if ctx.size <= m: m = 1
    d = np.abs(ctx[m:]-ctx[:-m]); s = d.mean() if d.size else 1.0
    return s if s>1e-8 else 1.0

def metrics(truth, pred, ctx, m):
    truth=np.asarray(truth,float); pred=np.asarray(pred,float); e=pred-truth
    mae=np.mean(np.abs(e)); rmse=np.sqrt(np.mean(e**2))
    den=np.abs(truth)+np.abs(pred); safe=np.where(den==0,1.0,den)
    smape=np.mean(np.where(den==0,0.0,2*np.abs(e)/safe))*100
    return {"MAE":mae,"RMSE":rmse,"sMAPE":smape,"MASE":mae/_scale(ctx,m)}

_t=np.array([1.,2.,3.]); _c=np.arange(50.0)
assert metrics(_t,_t,_c,1)["MAE"]==0 and metrics(_t,_t+1,_c,1)["MAE"]==1.0
print("metrics OK")


metrics OK


## 5. The three foundation models (uniform interface)

Each model exposes `predict(context, horizon) -> point forecast`; weights load once.


In [8]:
from abc import ABC, abstractmethod
class FM(ABC):
    name="fm"
    @abstractmethod
    def predict(self, context, horizon): ...
    def warmup(self): return self
    def free(self):
        """Release GPU memory held by this model (important on small VRAM)."""
        import gc
        for a in ("pipe","model","module"):
            if getattr(self, a, None) is not None: setattr(self, a, None)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

class ChronosFM(FM):
    name="chronos"
    def __init__(self): self.pipe=None
    def warmup(self):
        from chronos import BaseChronosPipeline
        dev="cuda" if torch.cuda.is_available() else "cpu"
        dt=torch.bfloat16 if dev=="cuda" else torch.float32
        self.pipe=BaseChronosPipeline.from_pretrained(CHRONOS_MODEL, device_map=dev, torch_dtype=dt)
        return self
    def predict(self, context, horizon):
        if self.pipe is None: self.warmup()
        ctx=torch.tensor(np.asarray(context[-CONTEXT_LENGTH:],float),dtype=torch.float32)
        q,_=self.pipe.predict_quantiles(context=ctx, prediction_length=horizon,
                                        quantile_levels=[0.1,0.5,0.9])
        return q[0,:,1].cpu().numpy().astype(float)

class TimesFMFM(FM):
    name="timesfm"
    def __init__(self): self.model=None
    def warmup(self):
        import timesfm
        self._ctx=(CONTEXT_LENGTH//32)*32
        backend="gpu" if torch.cuda.is_available() else "cpu"   # auto: works locally on CPU too
        self.model=timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(backend=backend, per_core_batch_size=16,
                horizon_len=max(g["horizon"] for g in GRANULARITIES.values()), context_len=self._ctx),
            checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id=TIMESFM_REPO))
        return self
    def predict(self, context, horizon):
        if self.model is None: self.warmup()
        pt,_=self.model.forecast([np.asarray(context[-self._ctx:],float)], freq=[0])
        return np.asarray(pt[0][:horizon],float)

class MoiraiFM(FM):
    name="moirai"
    def __init__(self,num_samples=100,patch_size=32): self.ns=num_samples; self.ps=patch_size; self.module=None
    def warmup(self):
        from uni2ts.model.moirai import MoiraiModule
        self.module=MoiraiModule.from_pretrained(MOIRAI_REPO); return self
    def predict(self, context, horizon):
        from uni2ts.model.moirai import MoiraiForecast
        if self.module is None: self.warmup()
        ctx=np.asarray(context[-CONTEXT_LENGTH:],float)
        model=MoiraiForecast(module=self.module, prediction_length=horizon, context_length=len(ctx),
            patch_size=self.ps, num_samples=self.ns, target_dim=1,
            feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0)
        dev="cuda" if torch.cuda.is_available() else "cpu"; model=model.to(dev)
        pt=torch.tensor(ctx,dtype=torch.float32,device=dev).reshape(1,-1,1)
        obs=torch.ones_like(pt,dtype=torch.bool); pad=torch.zeros(1,pt.shape[1],dtype=torch.bool,device=dev)
        with torch.no_grad():
            fc=model(past_target=pt, past_observed_target=obs, past_is_pad=pad)
        return np.median(fc[0].cpu().numpy(),axis=0)[:horizon].astype(float)

FM_CLASSES={"chronos":ChronosFM,"timesfm":TimesFMFM,"moirai":MoiraiFM}
# Instantiate (but DON'T load weights yet). On a 6 GB GPU we load one model at a
# time in the evaluation loop and free its VRAM before the next — see next section.
MODELS={k:FM_CLASSES[k]() for k,on in RUN_MODELS.items() if on}
print("models to run (loaded lazily, one at a time):", list(MODELS))


models to run (loaded lazily, one at a time): ['chronos', 'timesfm', 'moirai']


## 6. Run the models across all granularities

**Memory-safe for 6 GB GPUs.** Models are evaluated **one at a time**: each is loaded, run
over every window, then its GPU memory is freed before the next model loads. This keeps only
one model resident at a time, so an RTX 3060 (6 GB) won't run out of VRAM even with
TimesFM-500M in the mix.


In [9]:
import time, os
import pandas as pd
from tqdm.auto import tqdm

# Reduce fragmentation OOM on small GPUs (harmless elsewhere).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def gpu_mem():
    if torch.cuda.is_available():
        return f"{torch.cuda.memory_allocated()/1e9:.2f} GB used"
    return "CPU"

rows=[]
for name, model in MODELS.items():
    try:
        print(f"loading {name} ...", flush=True); model.warmup()
    except Exception as e:
        print(f"  SKIPPED {name}: {type(e).__name__}: {e}"); continue
    for w in tqdm(WINDOWS, desc=name):
        t0=time.perf_counter()
        try: p=model.predict(w.context,w.horizon); ok=True
        except Exception: p=None; ok=False
        dt=time.perf_counter()-t0
        met=(metrics(w.truth,p,w.context,w.m) if ok and p is not None and len(p)==w.horizon
             and np.all(np.isfinite(p)) else {"MAE":np.nan,"RMSE":np.nan,"sMAPE":np.nan,"MASE":np.nan})
        rows.append({"etype":w.etype,"gran":w.gran,"gran_label":GRANULARITIES[w.gran]["label"],
                     "series_id":w.series_id,"horizon":w.horizon,"model":name,"infer_sec":dt,**met})
    print(f"  {name} done | {gpu_mem()} -> freeing")
    model.free()

res=pd.DataFrame(rows); res.to_parquet(RESULTS_PATH,index=False)
print(len(res),"rows saved ->",RESULTS_PATH); res.head()


loading chronos ...


chronos: 100%|██████████| 54/54 [00:00<00:00, 12571.04it/s]

  chronos done | 0.10 GB used -> freeing
loading timesfm ...
  SKIPPED timesfm: AttributeError: module 'timesfm' has no attribute 'TimesFm'
loading moirai ...



moirai: 100%|██████████| 54/54 [00:03<00:00, 17.35it/s]


  moirai done | 0.07 GB used -> freeing
108 rows saved -> stage1_granularity_results.parquet


,etype,gran,gran_label,series_id,horizon,model,infer_sec,MAE,RMSE,sMAPE,MASE
0,load,native,minutes (native),0,48,chronos,0.002718,NaN,NaN,NaN,NaN
1,load,native,minutes (native),0,48,chronos,0.000054,NaN,NaN,NaN,NaN
2,load,1h,hourly,0,24,chronos,0.000014,NaN,NaN,NaN,NaN
3,load,1h,hourly,0,24,chronos,0.000010,NaN,NaN,NaN,NaN
4,load,1D,daily,0,14,chronos,0.000009,NaN,NaN,NaN,NaN


## 7. Compare the three models

### 7.1 Accuracy by granularity (per model)
Mean MASE at each granularity — the core "how do they behave across minutes/hours/day" table.


In [11]:
gran_order=[GRANULARITIES[k]["label"] for k in GRANULARITIES]
t=res.pivot_table(index="gran_label",columns="model",values="MASE",aggfunc="mean").reindex(gran_order)
t.style.format("{:.3f}").set_caption("Mean MASE by granularity (lower = better)")


model,moirai
gran_label,
minutes (native),1.356
hourly,1.365
daily,1.059


### 7.2 Overall model ranking

In [12]:
overall=res.groupby("model")[["MASE","sMAPE","RMSE","MAE","infer_sec"]].mean().sort_values("MASE")
print("Best overall (mean MASE):", overall.index[0] if len(overall) else None)
overall.style.format("{:.3f}").background_gradient(cmap="RdYlGn_r",subset=["MASE","sMAPE"])


Best overall (mean MASE): moirai


,MASE,sMAPE,RMSE,MAE,infer_sec
model,,,,,
moirai,1.260,74.039,102.636,84.063,2.931
chronos,nan,nan,nan,nan,0.003


### 7.3 Best model per energy type × granularity
Shows whether the winner changes with resolution / series type.


In [ ]:
if res["model"].nunique()>1:
    g=res.groupby(["etype","gran_label","model"])["MASE"].mean().reset_index()
    best=g.loc[g.groupby(["etype","gran_label"])["MASE"].idxmin()]
    display(best.pivot(index="etype",columns="gran_label",values="model")
            .reindex(columns=gran_order).style.set_caption("Best (lowest-MASE) model"))
    key=["etype","gran","series_id","horizon"]
    wide=res.pivot_table(index=key,columns="model",values="MASE")
    wr=wide.idxmin(axis=1).value_counts(normalize=True).rename("win_rate")
    display(wr.to_frame().style.format("{:.1%}").set_caption("Win-rate (fraction of windows a model is best)"))
else:
    print("Only one model active — enable all three in RUN_MODELS for the comparison.")


### 7.4 Visual: MASE vs granularity, per energy type

In [ ]:
import matplotlib.pyplot as plt
palette={"chronos":"#2563eb","timesfm":"#dc2626","moirai":"#7c3aed"}
types=list(DATA.keys())
fig,axes=plt.subplots(1,len(types),figsize=(5*len(types),4),squeeze=False)
for ax,et in zip(axes[0],types):
    sub=res[res.etype==et]
    for mdl in sub.model.unique():
        gg=sub[sub.model==mdl].groupby("gran_label")["MASE"].mean().reindex(gran_order)
        ax.plot(range(len(gg)),gg.values,marker="o",label=mdl,color=palette.get(mdl))
    ax.set_xticks(range(len(gran_order))); ax.set_xticklabels(gran_order,rotation=25,ha="right")
    ax.set_title(BASE[et]["label"]); ax.set_ylabel("mean MASE"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.suptitle("Foundation-model accuracy across granularities (minutes → hours → day)",y=1.03)
plt.tight_layout(); plt.show()


### 7.5 Visual: overall MASE per model (bar)

In [ ]:
ax=overall["MASE"].plot(kind="bar",figsize=(6,4),color=[palette.get(m,"#333") for m in overall.index])
ax.set_ylabel("mean MASE"); ax.set_title("Overall accuracy — three foundation models")
ax.grid(alpha=.3,axis="y"); plt.tight_layout(); plt.show()


### 7.6 One qualitative forecast (example window)

In [ ]:
ex=next((w for w in WINDOWS if w.gran=="1h"), WINDOWS[0])
fig,ax=plt.subplots(figsize=(11,4))
hist=ex.context[-3*ex.horizon:]
ax.plot(range(-len(hist),0),hist,color="#334155",lw=1,label="context")
ax.plot(range(0,ex.horizon),ex.truth,color="black",lw=2.2,label="truth")
for name,m in MODELS.items():
    try: ax.plot(range(0,ex.horizon),m.predict(ex.context,ex.horizon),lw=1.6,color=palette.get(name),label=name)
    except Exception as e: print("skip",name,e)
    finally: m.free()   # keep only one model in VRAM at a time (6 GB-safe)
ax.axvline(0,color="#cbd5e1",ls=":"); ax.legend(ncol=4,fontsize=8)
ax.set_title(f"Example — {BASE[ex.etype]['label']} @ {GRANULARITIES[ex.gran]['label']} (h={ex.horizon})")
ax.set_xlabel("steps (0 = forecast origin)"); plt.tight_layout(); plt.show()


## 8. Export

Saves tidy CSVs for the thesis appendix.


In [ ]:
res.to_csv("stage1_all_results.csv",index=False)
overall.to_csv("stage1_model_ranking.csv")
t.to_csv("stage1_mase_by_granularity.csv")
print("Saved: stage1_all_results.csv, stage1_model_ranking.csv, stage1_mase_by_granularity.csv")


### Notes for the write-up

- **No baselines** here by design; the seasonal-naïve term inside MASE is only a scale
  normaliser, not a competing model.
- Granularity is studied by **resampling one source** (minutes → hours → day), so
  differences are attributable to resolution, not to different signals. State the
  aggregation used (`mean`).
- Report both the **overall ranking** (§7.2) and the **best-model-per-condition** grid
  (§7.3): the interesting finding is usually that the winner changes with resolution and
  energy type, not that one model dominates everywhere.
- Start small (`SERIES_PER_TYPE=3`), confirm end-to-end, then scale up for final numbers.
